# Atelier 1.D — Extraction structurée ★

**Durée : 50 min** · Projet fil rouge, incrément du Jour 1

Pour traiter une demande, l'assistant doit en extraire les informations clés : numéros de client, de commande et de facture, produits, montants.
Vous définissez des **schémas Pydantic**, obtenez des sorties conformes et mesurez l'exactitude **champ par champ** sur des emails, des factures et des contrats.

In [ ]:
import outils_llm as llm

MODELE = llm.choisir_modele()  # ou forcez, par ex. "groq:openai/gpt-oss-120b", "nvidia:nvidia/nemotron-3.5-lightning-30b-a3b"
N_MAX = 20                     # les paliers gratuits limitent le débit ; None pour tout le jeu de test
print("Modèle utilisé :", MODELE)

import json
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field

VERITE = llm.charger_json("verite_extraction.json")
df = llm.charger_demandes()

## 1. Pourquoi imposer un schéma ?

Premier essai : demander « du JSON » dans le prompt, sans schéma.

In [ ]:
demande = df[df.id == next(iter(VERITE["emails"]))].iloc[0]
r = llm.demander(
    "Extrais les informations clés de cette demande au format JSON.\n\n" + llm.formater_demande(demande),
    modele=MODELE,
)
print(r.texte)
try:
    json.loads(r.texte)
    print("\n-> JSON valide… mais avec quels noms de champs ? Relancez la cellule et comparez.")
except json.JSONDecodeError as erreur:
    print("\n-> JSON invalide :", erreur)

Même quand le JSON est valide, les noms de champs et les formats changent d'un appel à l'autre : impossible de brancher un système dessus.
La solution : décrire la sortie attendue par un **schéma**, que l'API impose au modèle quand elle le permet (*structured outputs*), puis valider le résultat avec Pydantic.

Tous les modèles ne permettent pas d'imposer un schéma (chez Groq, par exemple, gpt-oss oui, Llama non). Dans ce cas, `llm.demander` décrit le schéma dans le prompt et valide la réponse : l'attribut `structure` de la réponse indique la méthode utilisée.

Les consignes communes à toutes les extractions :

In [ ]:
SYSTEME = """Tu extrais des informations de documents commerciaux de Fictiva SA.
Règles :
- Recopie les identifiants exactement comme dans le document.
- N'invente rien : si une information est absente, renvoie null (ou une liste vide).
- Montants en CHF, sous forme de nombres (1'007.00 devient 1007.0).
- Dates au format AAAA-MM-JJ."""

def extraire(texte, schema):
    return llm.demander(texte, systeme=SYSTEME, modele=MODELE, schema=schema)

## 2. Schéma pour les emails

Les champs attendus (voir `donnees/verite_extraction.json`) : `numero_client`, `numero_commande`, `numero_facture`, `entreprise`, `interlocuteur`, `produits` (références avec quantités) et `montant_chf`.

Les descriptions (`Field(description=...)`) font partie du prompt : elles lèvent les ambiguïtés. Par exemple, quel montant retenir quand l'email en cite deux ?

In [ ]:
class LigneProduit(BaseModel):
    reference: str = Field(description="Référence produit Fictiva, par ex. KB-80")
    quantite: int = Field(description="Quantité citée, 1 si non précisée")

class DemandeExtraite(BaseModel):
    numero_client: str | None = Field(description="Format CL-xxxxx, null si absent")
    numero_commande: str | None = Field(description="Format CMD-AAAA-xxxx, null si absent")
    numero_facture: str | None = Field(description="Format FAC-AAAA-xxxx, null si absent")
    entreprise: str | None = Field(description="Raison sociale du client")
    interlocuteur: str | None = Field(description="Nom de la personne qui écrit, null si non signé par une personne")
    produits: list[LigneProduit] = Field(description="Produits cités avec une référence Fictiva")
    montant_chf: float | None = Field(
        description="Montant que le client demande à recevoir ou qu'il conteste "
        "(remboursement, acompte, trop-perçu, facture contestée). null si aucun."
    )

In [ ]:
r = extraire(llm.formater_demande(demande), DemandeExtraite)
print("Méthode :", r.structure)
print(r.objet.model_dump_json(indent=2))

## 3. Mesurer champ par champ

La fonction `comparer` normalise les valeurs (casse, espaces, nombres) avant de les comparer à la vérité terrain.

In [ ]:
def normaliser(valeur):
    if isinstance(valeur, bool) or valeur is None:
        return valeur
    if isinstance(valeur, (int, float)):
        return round(float(valeur), 2)
    if isinstance(valeur, str):
        return " ".join(valeur.split()).casefold()
    if isinstance(valeur, list):
        return sorted(json.dumps(normaliser(v), sort_keys=True, ensure_ascii=False) for v in valeur)
    if isinstance(valeur, dict):  # le libellé libre des lignes n'est pas comparé
        return {cle: normaliser(v) for cle, v in valeur.items() if cle != "designation"}
    return valeur

def comparer(predit, attendu):
    if predit is None:
        return {champ: False for champ in attendu}
    obtenu = predit.model_dump()
    return {champ: normaliser(obtenu.get(champ)) == normaliser(valeur) for champ, valeur in attendu.items()}

def evaluer(objets, verite):
    """objets : {document: objet Pydantic ou None} ; renvoie le tableau des champs justes."""
    resultat = pd.DataFrame({doc: comparer(objets.get(doc), attendu) for doc, attendu in verite.items()}).T
    print(f"Exactitude globale : {resultat.values.mean():.0%}")
    print("Par champ :", resultat.mean().round(2).to_dict())
    return resultat.replace({True: "✓", False: "✗"})

def objets_de(reponses):
    return [r.objet if isinstance(r, llm.Reponse) else None for r in reponses]

def afficher_ecarts(objets, verite):
    """Valeur obtenue et valeur attendue pour chaque champ faux."""
    for doc, objet in objets.items():
        obtenu = objet.model_dump() if objet else {}
        ecarts = {champ: f"obtenu {obtenu.get(champ)!r} / attendu {attendu!r}"
                  for champ, attendu in verite[doc].items()
                  if not comparer(objet, {champ: attendu})[champ]}
        if ecarts:
            print(doc, ecarts, "\n")

In [ ]:
emails = {ident: df[df.id == ident].iloc[0] for ident in VERITE["emails"]}
reponses = llm.en_lot(lambda ligne: extraire(llm.formater_demande(ligne), DemandeExtraite), emails.values())
objets_emails = dict(zip(emails, objets_de(reponses)))
evaluer(objets_emails, VERITE["emails"])

Examinez les ✗ : erreur du modèle (identifiant inventé, avoir pris pour une facture), description de champ ambiguë, ou vérité terrain discutable ?

✏️ Ajustez les descriptions de champs et relancez.

In [ ]:
afficher_ecarts(objets_emails, VERITE["emails"])

## 4. Factures

Les cinq factures de `donnees/factures/` ont des mises en page différentes : tableau aligné, liste, tableau Markdown, document « scanné » avec séparateurs de milliers suisses (`1'007.00`).

In [ ]:
class LigneFacture(BaseModel):
    reference: str
    designation: str
    quantite: int
    prix_unitaire_chf: float
    total_ligne_chf: float

class Facture(BaseModel):
    numero_facture: str
    date_facture: str = Field(description="AAAA-MM-JJ")
    client_nom: str
    numero_client: str | None
    lignes: list[LigneFacture]
    remise_chf: float | None = Field(description="Montant de la remise en valeur positive, null si aucune")
    total_ht_chf: float = Field(description="Total hors taxes, après remise")
    taux_tva_pct: float
    montant_tva_chf: float
    total_ttc_chf: float
    delai_paiement_jours: int | None

In [ ]:
dossier = llm.DOSSIER_DONNEES / "factures"
textes = {nom: (dossier / nom).read_text(encoding="utf-8") for nom in VERITE["factures"]}
reponses = llm.en_lot(lambda texte: extraire(texte, Facture), textes.values())
objets_factures = dict(zip(textes, objets_de(reponses)))
evaluer(objets_factures, VERITE["factures"])

In [ ]:
afficher_ecarts(objets_factures, VERITE["factures"])

## 5. Contrats

Même démarche pour les trois contrats de `donnees/contrats/`. Ici, les informations sont dans des phrases (« pour une durée de trois (3) ans », « huit cent cinquante francs ») : le modèle doit interpréter, pas seulement recopier.
`Literal` restreint `type_contrat` et `periodicite` aux valeurs autorisées.

In [ ]:
class Contrat(BaseModel):
    type_contrat: Literal["fourniture", "maintenance", "location", "autre"]
    client_nom: str
    date_debut: str = Field(description="AAAA-MM-JJ")
    duree_mois: int = Field(description="Durée initiale, en mois")
    reconduction_tacite: bool
    preavis_resiliation_mois: int | None
    montant_chf: float = Field(description="Montant principal HT : redevance, loyer ou engagement d'achat")
    periodicite: Literal["mensuel", "annuel", "unique"] = Field(description="Périodicité de ce montant")
    delai_paiement_jours: int | None

In [ ]:
dossier = llm.DOSSIER_DONNEES / "contrats"
textes = {nom: (dossier / nom).read_text(encoding="utf-8") for nom in VERITE["contrats"]}
reponses = llm.en_lot(lambda texte: extraire(texte, Contrat), textes.values())
objets_contrats = dict(zip(textes, objets_de(reponses)))
evaluer(objets_contrats, VERITE["contrats"])

## 6. Valider au-delà du schéma

Un schéma garantit la **forme**, pas la **justesse**. Des règles métier simples détectent une partie des erreurs : sommes des lignes, remise, TVA, total TTC.

In [ ]:
def controler_facture(f):
    alertes = []
    for ligne in f.lignes:
        if abs(ligne.quantite * ligne.prix_unitaire_chf - ligne.total_ligne_chf) > 0.01:
            alertes.append(f"ligne {ligne.reference} : {ligne.quantite} × {ligne.prix_unitaire_chf} ≠ {ligne.total_ligne_chf}")
    net = sum(ligne.total_ligne_chf for ligne in f.lignes) - (f.remise_chf or 0)
    if abs(net - f.total_ht_chf) > 0.01:
        alertes.append(f"lignes - remise = {net:.2f} ≠ total HT {f.total_ht_chf:.2f}")
    if abs(f.total_ht_chf * f.taux_tva_pct / 100 - f.montant_tva_chf) > 0.01:
        alertes.append("TVA incohérente avec le total HT")
    if abs(f.total_ht_chf + f.montant_tva_chf - f.total_ttc_chf) > 0.01:
        alertes.append("total TTC ≠ HT + TVA")
    return alertes

for nom, facture in objets_factures.items():
    alertes = controler_facture(facture) if facture else ["extraction échouée"]
    print(f"{nom} :", "OK" if not alertes else "; ".join(alertes))

**Questions**
1. Une alerte signale-t-elle une erreur d'extraction ou une erreur dans la facture elle-même ? Comment le savoir ?
2. La facture FAC-2026-1187 passe tous les contrôles. Pourtant l'Atelier Morel Sàrl la conteste par email : le prix facturé est de 650 CHF au lieu des 600 CHF de l'offre. Quelle information manque pour le détecter ?

## À retenir

- Schéma imposé + validation Pydantic = sortie exploitable par un programme. Quand l'API ne sait pas imposer le schéma, on le décrit dans le prompt et on valide.
- Les descriptions de champs sont des instructions : elles lèvent les ambiguïtés.
- On mesure champ par champ sur une vérité terrain, puis on ajoute des **contrôles métier** : le schéma ne garantit pas la justesse.